In [1]:
# imports and config
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import searoute as sr
import requests
import country_converter as coco
import folium
from IPython.display import display
pd.set_option("display.max_columns", 120)

In [2]:
dim_country = pd.read_parquet("/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/comtrade/dimensions/dim_country.parquet")

In [ ]:
# build dim_country_ports from World Port Index (max 2 representative ports per ISO3)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PORT_INDEX_PATH = PROJECT_ROOT / "ingest" / "Kaggle" / "world_port_index" / "UpdatedPub150.csv"
if not PORT_INDEX_PATH.exists():
    raise FileNotFoundError(f"World Port Index file not found: {PORT_INDEX_PATH}")

port_raw = pd.read_csv(PORT_INDEX_PATH, low_memory=False)

port_candidates = port_raw.rename(
    columns={
        "Main Port Name": "main_port_name",
        "Alternate Port Name": "alternate_port_name",
        "Country Code": "country_label",
        "Harbor Size": "harbor_size",
        "Latitude": "latitude",
        "Longitude": "longitude",
    }
).copy()

port_candidates["main_port_name"] = port_candidates["main_port_name"].astype("string").str.strip()
port_candidates["alternate_port_name"] = port_candidates["alternate_port_name"].astype("string").str.strip()
port_candidates["country_label"] = port_candidates["country_label"].astype("string").str.strip()

port_candidates["port_name"] = port_candidates["main_port_name"].fillna(port_candidates["alternate_port_name"])
port_candidates["latitude"] = pd.to_numeric(port_candidates["latitude"], errors="coerce")
port_candidates["longitude"] = pd.to_numeric(port_candidates["longitude"], errors="coerce")

# Convert country labels from the port file to ISO3 for joins with dim_country.
port_candidates["iso3"] = coco.convert(
    names=port_candidates["country_label"].fillna(""),
    to="ISO3",
    not_found="not found",
)
port_candidates["iso3"] = pd.Series(port_candidates["iso3"], index=port_candidates.index).replace("not found", pd.NA)

size_rank = {"Very Small": 1, "Small": 2, "Medium": 3, "Large": 4}
port_candidates["size_rank"] = port_candidates["harbor_size"].map(size_rank).fillna(0).astype(int)

usable_ports = (
    port_candidates
    .dropna(subset=["iso3", "port_name", "latitude", "longitude"])
    .drop_duplicates(subset=["iso3", "port_name", "latitude", "longitude"])
    .sort_values(["iso3", "size_rank", "port_name"], ascending=[True, False, True])
    .copy()
)

top_ports = usable_ports.groupby("iso3", group_keys=False).head(2).copy()
top_ports["port_rank"] = top_ports.groupby("iso3").cumcount() + 1

dim_country_base = dim_country[["iso3", "country_name"]].drop_duplicates().copy()
dim_country_ports = dim_country_base.merge(
    top_ports[["iso3", "port_rank", "port_name", "latitude", "longitude", "harbor_size"]],
    on="iso3",
    how="left",
)

display(dim_country_ports.head(20))
dim_country_ports.shape

Wake Island not found in regex
Johnson Atoll not found in regex
Midway Islands not found in regex
 not found in regex
 not found in regex
 not found in regex
 not found in regex


,iso3,country_name,port_rank,port_name,latitude,longitude,harbor_size
0,A79,"LAIA, nes",NaN,<NA>,NaN,NaN,NaN
1,ABW,Aruba,1.0,Sint Nicolaas Baai,12.433333,-69.916667,Small
2,ABW,Aruba,2.0,Paarden Baai - (Oranjestad),12.516667,-70.033333,Very Small
3,AFG,Afghanistan,NaN,<NA>,NaN,NaN,NaN
4,AGO,Angola,1.0,Cabinda,-5.533333,12.200000,Small
5,AGO,Angola,2.0,Estrela Oil Field,-6.450000,12.400000,Small
6,AIA,Anguilla,NaN,<NA>,NaN,NaN,NaN
7,ALB,Albania,1.0,Durres,41.316667,19.450000,Small
8,ALB,Albania,2.0,Shengjin,41.816667,19.600000,Very Small
9,AND,Andorra,NaN,<NA>,NaN,NaN,NaN


(390, 7)

In [12]:
dim_country_ports.shape

(390, 7)

In [13]:
# inspect available regional attributes in dim_country
region_like_cols = [c for c in dim_country.columns if any(k in c.lower() for k in ["region", "subregion", "continent", "area", "m49"])]
display(region_like_cols)
display(dim_country[region_like_cols].head(5))

['region', 'subregion', 'continent']

,region,subregion,continent
0,Americas,Latin America and the Caribbean,Americas
1,Americas,Caribbean,North America
2,Asia,Southern Asia,Asia
3,Africa,Middle Africa,Africa
4,Americas,Caribbean,North America


In [20]:
# quick coverage check for dim_country -> representative ports
country_port_coverage = (
    dim_country_ports.groupby(["iso3", "country_name"], as_index=False)["port_rank"]
    .count()
    .rename(columns={"port_rank": "ports_found"})
)

missing_country_ports = country_port_coverage[country_port_coverage["ports_found"] == 0].copy()
display(country_port_coverage.sort_values(["ports_found", "country_name"]).head(20))
missing_country_ports.shape
PROJECT_ROOT = Path("/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly")

analytics_path = Path(PROJECT_ROOT / "data/silver/comtrade/analytics")
analytics_path.mkdir(parents=True, exist_ok=True)
missing_country_ports.to_csv(analytics_path / "missing_country_ports.csv", index=False)


,iso3,country_name,ports_found
2,AFG,Afghanistan,0
6,AND,Andorra,0
4,AIA,Anguilla,0
238,_X,"Areas, nes",0
9,ARM,Armenia,0
15,AUT,Austria,0
16,AZE,Azerbaijan,0
28,BLR,Belarus,0
35,BTN,Bhutan,0
31,BOL,Bolivia (Plurinational State of),0


In [ ]:
# paths, chokepoints, and routing metadata
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

FACT_ROOT = PROJECT_ROOT / "data" / "silver" / "comtrade" / "comtrade_fact"
FACT_FILES = sorted(FACT_ROOT.rglob("ref_year=*/reporter_iso3=*/*.parquet"))

DIMENSIONS_ROOT = PROJECT_ROOT / "data" / "silver" / "comtrade" / "dimensions"
DIMENSIONS_ROOT.mkdir(parents=True, exist_ok=True)

DIM_OUTPUT_PATH = PROJECT_ROOT / "data" / "silver" / "comtrade" / "dim_trade_routes.parquet"
COUNTRY_ZONE_PATH = DIMENSIONS_ROOT / "dim_country_maritime_zone.parquet"
CHOKEPOINT_DIM_PATH = DIMENSIONS_ROOT / "dim_chokepoint.parquet"
CHOKEPOINT_RULES_PATH = DIMENSIONS_ROOT / "bridge_maritime_zone_chokepoints.parquet"

# Build per-country port candidates from dim_country_ports with searoute-ready (lon, lat) ordering.
country_ports_clean = (
    dim_country_ports
    .dropna(subset=["iso3", "port_name", "latitude", "longitude"])
)

COUNTRY_PORTS = (
    country_ports_clean.sort_values(["iso3", "port_rank", "port_name"])
    .groupby("iso3")
    .apply(
        lambda g: [
            {
                "port_name": row.port_name,
                "lonlat": (float(row.longitude), float(row.latitude)),
            }
            for row in g.itertuples(index=False)
        ]
    )
    .to_dict()
)

# Use dim_country attributes directly and persist a reusable maritime-zone dimension.
country_regions = dim_country[["iso3", "country_name", "region", "subregion", "continent"]].drop_duplicates().copy()
country_regions["iso3"] = country_regions["iso3"].astype("string").str.upper()

def _as_txt(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

def _to_maritime_zone(row) -> str:
    region = _as_txt(row.region)
    subregion = _as_txt(row.subregion)
    continent = _as_txt(row.continent)
    country_name = _as_txt(row.country_name)

    if "western asia" in subregion or any(k in country_name for k in ["saudi", "emirates", "qatar", "kuwait", "oman", "bahrain"]):
        return "GULF"
    if "eastern asia" in subregion or "south-eastern asia" in subregion or "southern asia" in subregion:
        return "ASIA"
    if "europe" in region or "europe" in continent:
        return "EUROPE"
    if "africa" in region or "africa" in continent:
        return "AFRICA"
    if "americas" in region or "america" in continent:
        return "AMERICAS"
    if "oceania" in region or "oceania" in continent or "australia" in continent:
        return "OCEANIA"
    return "OTHER"

country_regions["maritime_zone"] = country_regions.apply(_to_maritime_zone, axis=1)
country_regions.to_parquet(COUNTRY_ZONE_PATH, index=False)

# Persist chokepoint master data as a dimension table.
dim_chokepoint = pd.DataFrame(
    [
        {"chokepoint_name": "Suez Canal", "longitude": 32.5498, "latitude": 30.8167, "kind": "canal"},
        {"chokepoint_name": "Hormuz Strait", "longitude": 56.25, "latitude": 26.57, "kind": "strait"},
        {"chokepoint_name": "Cape of Good Hope", "longitude": 18.47, "latitude": -34.36, "kind": "cape"},
        {"chokepoint_name": "Bab el-Mandeb", "longitude": 43.33, "latitude": 12.58, "kind": "strait"},
        {"chokepoint_name": "Panama Canal", "longitude": -79.58, "latitude": 9.08, "kind": "canal"},
        {"chokepoint_name": "Malacca Strait", "longitude": 99.8, "latitude": 2.5, "kind": "strait"},
        {"chokepoint_name": "Gibraltar Strait", "longitude": -5.6, "latitude": 35.95, "kind": "strait"},
    ]
)
dim_chokepoint.to_parquet(CHOKEPOINT_DIM_PATH, index=False)

# Persist route forcing rules as a bridge table so notebook logic reads from data, not literals.
bridge_maritime_zone_chokepoints = pd.DataFrame(
    [
        {"origin_zone": "EUROPE", "destination_zone": "ASIA", "leg_order": 1, "chokepoint_name": "Suez Canal"},
        {"origin_zone": "ASIA", "destination_zone": "EUROPE", "leg_order": 1, "chokepoint_name": "Suez Canal"},
        {"origin_zone": "EUROPE", "destination_zone": "GULF", "leg_order": 1, "chokepoint_name": "Suez Canal"},
        {"origin_zone": "EUROPE", "destination_zone": "GULF", "leg_order": 2, "chokepoint_name": "Bab el-Mandeb"},
        {"origin_zone": "EUROPE", "destination_zone": "GULF", "leg_order": 3, "chokepoint_name": "Hormuz Strait"},
        {"origin_zone": "GULF", "destination_zone": "EUROPE", "leg_order": 1, "chokepoint_name": "Hormuz Strait"},
        {"origin_zone": "GULF", "destination_zone": "EUROPE", "leg_order": 2, "chokepoint_name": "Bab el-Mandeb"},
        {"origin_zone": "GULF", "destination_zone": "EUROPE", "leg_order": 3, "chokepoint_name": "Suez Canal"},
        {"origin_zone": "ASIA", "destination_zone": "GULF", "leg_order": 1, "chokepoint_name": "Hormuz Strait"},
        {"origin_zone": "ASIA", "destination_zone": "GULF", "leg_order": 2, "chokepoint_name": "Malacca Strait"},
        {"origin_zone": "GULF", "destination_zone": "ASIA", "leg_order": 1, "chokepoint_name": "Hormuz Strait"},
        {"origin_zone": "GULF", "destination_zone": "ASIA", "leg_order": 2, "chokepoint_name": "Malacca Strait"},
        {"origin_zone": "EUROPE", "destination_zone": "AFRICA", "leg_order": 1, "chokepoint_name": "Gibraltar Strait"},
        {"origin_zone": "AFRICA", "destination_zone": "EUROPE", "leg_order": 1, "chokepoint_name": "Gibraltar Strait"},
        {"origin_zone": "AMERICAS", "destination_zone": "ASIA", "leg_order": 1, "chokepoint_name": "Panama Canal"},
        {"origin_zone": "AMERICAS", "destination_zone": "ASIA", "leg_order": 2, "chokepoint_name": "Malacca Strait"},
        {"origin_zone": "ASIA", "destination_zone": "AMERICAS", "leg_order": 1, "chokepoint_name": "Malacca Strait"},
        {"origin_zone": "ASIA", "destination_zone": "AMERICAS", "leg_order": 2, "chokepoint_name": "Panama Canal"},
        {"origin_zone": "EUROPE", "destination_zone": "OCEANIA", "leg_order": 1, "chokepoint_name": "Suez Canal"},
        {"origin_zone": "EUROPE", "destination_zone": "OCEANIA", "leg_order": 2, "chokepoint_name": "Malacca Strait"},
        {"origin_zone": "OCEANIA", "destination_zone": "EUROPE", "leg_order": 1, "chokepoint_name": "Malacca Strait"},
        {"origin_zone": "OCEANIA", "destination_zone": "EUROPE", "leg_order": 2, "chokepoint_name": "Suez Canal"},
    ]
)
bridge_maritime_zone_chokepoints.to_parquet(CHOKEPOINT_RULES_PATH, index=False)

# Load in-memory routing dictionaries from persisted dimensions.
dim_country_maritime_zone = pd.read_parquet(COUNTRY_ZONE_PATH)
dim_chokepoint = pd.read_parquet(CHOKEPOINT_DIM_PATH)
bridge_maritime_zone_chokepoints = pd.read_parquet(CHOKEPOINT_RULES_PATH)

ISO3_REGION = dict(zip(dim_country_maritime_zone["iso3"], dim_country_maritime_zone["maritime_zone"]))
CHOKEPOINT_COORDS = {
    row.chokepoint_name: (float(row.longitude), float(row.latitude))
    for row in dim_chokepoint.itertuples(index=False)
}
REGION_CHOKEPOINTS = (
    bridge_maritime_zone_chokepoints
    .sort_values(["origin_zone", "destination_zone", "leg_order"])
    .groupby(["origin_zone", "destination_zone"])["chokepoint_name"]
    .agg(list)
    .to_dict()
)

# Fallback pair hints remain useful for specific exceptions not yet encoded in the bridge table.
CHOKEPOINT_PAIR_HINTS = [
    ({"ESP", "CHN"}, "Suez Canal"),
    ({"ROU", "CHN"}, "Suez Canal"),
    ({"NLD", "USA"}, "Atlantic"),
    ({"SAU", "ARE"}, "Hormuz Strait"),
    ({"QAT", "CHN"}, "Hormuz Strait"),
    ({"ZAF", "CHN"}, "Cape of Good Hope"),
]

display(dim_country_maritime_zone[["maritime_zone"]].value_counts().rename("country_count").head(10))
display(dim_chokepoint)
display(bridge_maritime_zone_chokepoints.head(10))
str(COUNTRY_ZONE_PATH), str(CHOKEPOINT_DIM_PATH), str(CHOKEPOINT_RULES_PATH)

maritime_zone
AFRICA           57
AMERICAS         51
EUROPE           45
ASIA             27
OCEANIA          25
GULF             19
OTHER            15
Name: country_count, dtype: int64

,chokepoint_name,longitude,latitude,kind
0,Suez Canal,32.5498,30.8167,canal
1,Hormuz Strait,56.2500,26.5700,strait
2,Cape of Good Hope,18.4700,-34.3600,cape
3,Bab el-Mandeb,43.3300,12.5800,strait
4,Panama Canal,-79.5800,9.0800,canal
5,Malacca Strait,99.8000,2.5000,strait
6,Gibraltar Strait,-5.6000,35.9500,strait


,origin_zone,destination_zone,leg_order,chokepoint_name
0,EUROPE,ASIA,1,Suez Canal
1,ASIA,EUROPE,1,Suez Canal
2,EUROPE,GULF,1,Suez Canal
3,EUROPE,GULF,2,Bab el-Mandeb
4,EUROPE,GULF,3,Hormuz Strait
5,GULF,EUROPE,1,Hormuz Strait
6,GULF,EUROPE,2,Bab el-Mandeb
7,GULF,EUROPE,3,Suez Canal
8,ASIA,GULF,1,Hormuz Strait
9,ASIA,GULF,2,Malacca Strait


('/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/comtrade/dimensions/dim_country_maritime_zone.parquet',
 '/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/comtrade/dimensions/dim_chokepoint.parquet',
 '/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/comtrade/dimensions/bridge_maritime_zone_chokepoints.parquet')

In [6]:
# raw load + route key extraction
if not FACT_FILES:
    raise FileNotFoundError(f"No parquet fact files found under: {FACT_ROOT}")

silver_df = pd.concat([pd.read_parquet(path) for path in FACT_FILES], ignore_index=True)

silver_grain = ["period", "reporter_iso3", "partner_iso3", "flowCode", "cmdCode", "classification_version","customsCode", "motCode", "partner2Code"]
silver_df.drop_duplicates(subset=silver_grain, inplace=True)

def _normalize_iso3(series: pd.Series) -> pd.Series:
    clean = series.astype("string").str.upper().str.strip()
    clean = clean.replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "W00": pd.NA})
    valid = clean.str.fullmatch(r"[A-Z]{3}", na=False)
    return clean.where(valid, pd.NA)

if "partner2_iso3" in silver_df.columns:
    partner2_col = "partner2_iso3"
elif "partner2ISO" in silver_df.columns:
    partner2_col = "partner2ISO"
else:
    partner2_col = None

routes = silver_df[["reporter_iso3", "partner_iso3"]].copy()
if partner2_col:
    routes["partner2_iso3"] = silver_df[partner2_col]
else:
    routes["partner2_iso3"] = pd.NA

routes["reporter_iso3"] = _normalize_iso3(routes["reporter_iso3"])
routes["partner_iso3"] = _normalize_iso3(routes["partner_iso3"])
routes["partner2_iso3"] = _normalize_iso3(routes["partner2_iso3"])

routes = routes.dropna(subset=["reporter_iso3", "partner_iso3"]).drop_duplicates().reset_index(drop=True)
routes.head(), routes.shape

(  reporter_iso3 partner_iso3 partner2_iso3
 0           BGR          ALB          <NA>
 1           BGR          ARG          <NA>
 2           BGR          ARM          <NA>
 3           BGR          ATG          <NA>
 4           BGR          AUT          <NA>,
 (1488, 3))

In [18]:
# build dim_trade_routes using direct and chokepoint-forced routing
def _gc_distance_km(a_lonlat: tuple[float, float], b_lonlat: tuple[float, float]) -> float:
    lon1, lat1 = map(math.radians, a_lonlat)
    lon2, lat2 = map(math.radians, b_lonlat)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371.0088 * (2 * math.asin(math.sqrt(h)))

def _sea_distance_km(a_lonlat: tuple[float, float], b_lonlat: tuple[float, float]) -> float:
    route = sr.searoute(list(a_lonlat), list(b_lonlat), units="km")
    return float(route.properties["length"]), route

def _best_port_pair(a_iso: str, b_iso: str):
    a_ports = COUNTRY_PORTS.get(a_iso, [])
    b_ports = COUNTRY_PORTS.get(b_iso, [])
    if not a_ports or not b_ports:
        return np.nan, np.nan, None, None, []

    best_gc = np.inf
    best_a = None
    best_b = None
    for a in a_ports:
        for b in b_ports:
            gc_km = _gc_distance_km(a["lonlat"], b["lonlat"])
            if gc_km < best_gc:
                best_gc = gc_km
                best_a = a
                best_b = b

    sea_km = np.nan
    route_coords = []
    try:
        sea_km, sea_route = _sea_distance_km(best_a["lonlat"], best_b["lonlat"])
        route_coords = sea_route["geometry"]["coordinates"]
    except Exception:
        pass

    return best_gc, sea_km, best_a["port_name"], best_b["port_name"], route_coords

def _region_pair_chokepoints(r_iso: str, p_iso: str) -> list[str]:
    r_region = ISO3_REGION.get(r_iso, "OTHER")
    p_region = ISO3_REGION.get(p_iso, "OTHER")
    return REGION_CHOKEPOINTS.get((r_region, p_region), [])

def _fallback_pair_hint(r: str, p2: str | None, p: str) -> str:
    legs = [{r, p}] if not p2 else [{r, p2}, {p2, p}]
    for pair_set in legs:
        for hinted_pair, cp in CHOKEPOINT_PAIR_HINTS:
            if pair_set == hinted_pair:
                return cp
    return "Other / Unclassified"

def _forced_path_distance(start_lonlat: tuple[float, float], end_lonlat: tuple[float, float], chokepoints: list[str]):
    if not chokepoints:
        dist_km, route = _sea_distance_km(start_lonlat, end_lonlat)
        return dist_km, route["geometry"]["coordinates"]

    path_nodes = [start_lonlat] + [CHOKEPOINT_COORDS[cp] for cp in chokepoints if cp in CHOKEPOINT_COORDS] + [end_lonlat]
    if len(path_nodes) < 2:
        return np.nan, []

    total_km = 0.0
    stitched = []
    for i in range(len(path_nodes) - 1):
        leg_km, leg_route = _sea_distance_km(path_nodes[i], path_nodes[i + 1])
        total_km += leg_km
        leg_coords = leg_route["geometry"]["coordinates"]
        if i == 0:
            stitched.extend(leg_coords)
        else:
            stitched.extend(leg_coords[1:])
    return total_km, stitched

def _route_group(main_chokepoint: str) -> str:
    label = main_chokepoint.lower()
    if "suez" in label:
        return "SUEZ_EXPOSED"
    if "hormuz" in label:
        return "HORMUZ_EXPOSED"
    if "cape" in label:
        return "CAPE_DIVERSION_RISK"
    if "atlantic" in label:
        return "ATLANTIC_ROUTE"
    if "panama" in label:
        return "PANAMA_EXPOSED"
    if "malacca" in label:
        return "MALACCA_EXPOSED"
    if "gibraltar" in label:
        return "GIBRALTAR_EXPOSED"
    return "OTHER_ROUTE"

records = []
for row in routes.itertuples(index=False):
    r_iso = row.reporter_iso3
    p_iso = row.partner_iso3
    p2_iso = row.partner2_iso3 if pd.notna(row.partner2_iso3) else None

    if r_iso not in COUNTRY_PORTS or p_iso not in COUNTRY_PORTS:
        records.append(
            {
                "reporter_iso3": r_iso,
                "partner_iso3": p_iso,
                "partner2_iso3": p2_iso,
                "reporter_port": None,
                "partner2_port": None,
                "partner_port": None,
                "distance_km": np.nan,
                "sea_distance_km": np.nan,
                "sea_distance_direct_km": np.nan,
                "sea_distance_forced_km": np.nan,
                "main_chokepoint": "Missing port mapping",
                "route_group": "UNMAPPED",
                "route_mode": "unmapped",
                "route_path_coords": [],
            }
        )
        continue

    try:
        # Pick representative ports by shortest GC baseline.
        gc_km, sea_direct_km, reporter_port, partner_port, direct_coords = _best_port_pair(r_iso, p_iso)
        partner2_port = None

        cp_sequence = _region_pair_chokepoints(r_iso, p_iso)
        cp_hint = _fallback_pair_hint(r_iso, p2_iso, p_iso)
        if cp_hint in CHOKEPOINT_COORDS and cp_hint not in cp_sequence:
            cp_sequence = cp_sequence + [cp_hint]

        reporter_lonlat = next(p["lonlat"] for p in COUNTRY_PORTS[r_iso] if p["port_name"] == reporter_port)
        partner_lonlat = next(p["lonlat"] for p in COUNTRY_PORTS[p_iso] if p["port_name"] == partner_port)

        if cp_sequence:
            sea_forced_km, forced_coords = _forced_path_distance(reporter_lonlat, partner_lonlat, cp_sequence)
            sea_km = sea_forced_km if np.isfinite(sea_forced_km) else sea_direct_km
            main_cp = cp_sequence[0]
            route_mode = "forced_chokepoint"
            route_coords = forced_coords if forced_coords else direct_coords
        else:
            sea_km = sea_direct_km
            main_cp = cp_hint
            route_mode = "direct"
            route_coords = direct_coords

        # Preserve partner2 context for analytics while keeping main path reporter->partner.
        if p2_iso and p2_iso in COUNTRY_PORTS:
            _, _, _, partner2_port, _ = _best_port_pair(r_iso, p2_iso)

        records.append(
            {
                "reporter_iso3": r_iso,
                "partner_iso3": p_iso,
                "partner2_iso3": p2_iso,
                "reporter_port": reporter_port,
                "partner2_port": partner2_port,
                "partner_port": partner_port,
                "distance_km": round(gc_km, 2) if np.isfinite(gc_km) else np.nan,
                "sea_distance_km": round(sea_km, 2) if np.isfinite(sea_km) else np.nan,
                "sea_distance_direct_km": round(sea_direct_km, 2) if np.isfinite(sea_direct_km) else np.nan,
                "sea_distance_forced_km": round(sea_forced_km, 2) if cp_sequence and np.isfinite(sea_forced_km) else np.nan,
                "main_chokepoint": main_cp,
                "route_group": _route_group(main_cp),
                "route_mode": route_mode,
                "route_path_coords": route_coords,
            }
        )
    except Exception as exc:
        records.append(
            {
                "reporter_iso3": r_iso,
                "partner_iso3": p_iso,
                "partner2_iso3": p2_iso,
                "reporter_port": None,
                "partner2_port": None,
                "partner_port": None,
                "distance_km": np.nan,
                "sea_distance_km": np.nan,
                "sea_distance_direct_km": np.nan,
                "sea_distance_forced_km": np.nan,
                "main_chokepoint": f"Route error: {type(exc).__name__}",
                "route_group": "ERROR",
                "route_mode": "error",
                "route_path_coords": [],
            }
        )

dim_trade_routes = pd.DataFrame.from_records(records).drop_duplicates(
    subset=["reporter_iso3", "partner_iso3", "partner2_iso3"]
).sort_values(["reporter_iso3", "partner_iso3", "partner2_iso3"], na_position="last")

# Keep object-heavy path coordinates outside parquet for a slim persisted table.
dim_trade_routes_export = dim_trade_routes.drop(columns=["route_path_coords"]).copy()
DIM_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
dim_trade_routes_export.to_parquet(DIM_OUTPUT_PATH, index=False)

display(dim_trade_routes.head(20))
display(dim_trade_routes["route_mode"].value_counts(dropna=False))
display(dim_trade_routes["route_group"].value_counts(dropna=False))
str(DIM_OUTPUT_PATH)

,reporter_iso3,partner_iso3,partner2_iso3,reporter_port,partner2_port,partner_port,distance_km,sea_distance_km,sea_distance_direct_km,sea_distance_forced_km,main_chokepoint,route_group,route_mode,route_path_coords
1375,BGR,AGO,None,Burgas,None,Cabinda,5558.37,10422.16,10422.16,10422.16,Gibraltar Strait,GIBRALTAR_EXPOSED,forced_chokepoint,"[[27.6891, 42.4795], [28.0975, 42.4188], [29.1..."
0,BGR,ALB,None,Burgas,None,Shengjin,653.87,1572.86,1572.86,NaN,Other / Unclassified,OTHER_ROUTE,direct,"[[27.6891, 42.4795], [28.0975, 42.4188], [29.1..."
71,BGR,ARE,None,Burgas,None,Abu Zaby,3169.30,7275.16,7275.16,7275.16,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,"[[27.6891, 42.4795], [28.0975, 42.4188], [29.1..."
1,BGR,ARG,None,Burgas,None,La Plata,12196.58,13456.78,13456.78,NaN,Other / Unclassified,OTHER_ROUTE,direct,"[[27.6891, 42.4795], [28.0975, 42.4188], [29.1..."
2,BGR,ARM,None,NaN,None,NaN,NaN,NaN,NaN,NaN,Missing port mapping,UNMAPPED,unmapped,[]
3,BGR,ATG,None,Burgas,None,St Johns,8679.31,9524.19,9524.19,NaN,Other / Unclassified,OTHER_ROUTE,direct,"[[27.6891, 42.4795], [28.0975, 42.4188], [29.1..."
1382,BGR,AUS,None,Varna,None,Fremantle,12212.65,15425.87,13871.09,15425.87,Suez Canal,SUEZ_EXPOSED,forced_chokepoint,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
4,BGR,AUT,None,NaN,None,NaN,NaN,NaN,NaN,NaN,Missing port mapping,UNMAPPED,unmapped,[]
72,BGR,AZE,None,NaN,None,NaN,NaN,NaN,NaN,NaN,Missing port mapping,UNMAPPED,unmapped,[]
5,BGR,BEL,None,Varna,None,Antwerpen,1980.03,6136.35,6136.35,NaN,Other / Unclassified,OTHER_ROUTE,direct,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."


route_mode
direct               549
forced_chokepoint    479
unmapped             460
Name: count, dtype: int64

route_group
OTHER_ROUTE            547
UNMAPPED               460
SUEZ_EXPOSED           217
GIBRALTAR_EXPOSED      139
HORMUZ_EXPOSED          64
MALACCA_EXPOSED         37
PANAMA_EXPOSED          21
ATLANTIC_ROUTE           2
CAPE_DIVERSION_RISK      1
Name: count, dtype: int64

'/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/comtrade/dim_trade_routes.parquet'

In [19]:
# visualize a sample of computed routes on an interactive map
sample_size = 30
plot_df = dim_trade_routes[
    dim_trade_routes["route_path_coords"].map(lambda x: isinstance(x, list) and len(x) > 1)
].head(sample_size).copy()

if plot_df.empty:
    print("No route geometries available to plot.")
else:
    route_map = folium.Map(location=[20, 0], zoom_start=2, tiles="CartoDB positron")

    for row in plot_df.itertuples(index=False):
        coords_latlon = [[pt[1], pt[0]] for pt in row.route_path_coords]
        color = "#d62728" if row.route_mode == "forced_chokepoint" else "#1f77b4"
        popup = (
            f"{row.reporter_iso3} ({row.reporter_port}) -> {row.partner_iso3} ({row.partner_port})<br>"
            f"mode={row.route_mode}, cp={row.main_chokepoint}<br>"
            f"sea_km={row.sea_distance_km}"
        )
        folium.PolyLine(coords_latlon, color=color, weight=2.5, opacity=0.85, popup=popup).add_to(route_map)

        if coords_latlon:
            folium.CircleMarker(coords_latlon[0], radius=3, color="#2ca02c", fill=True).add_to(route_map)
            folium.CircleMarker(coords_latlon[-1], radius=3, color="#ff7f0e", fill=True).add_to(route_map)

    display(route_map)
    plot_df[[
        "reporter_iso3",
        "partner_iso3",
        "route_mode",
        "main_chokepoint",
        "sea_distance_direct_km",
        "sea_distance_forced_km",
        "sea_distance_km",
    ]].head(10)